# Phase 1 Task B — Results Driver

Sau khi cả 5 notebooks đã chạy xong, chạy notebook này
để tổng hợp kết quả từ wandb.

**Chạy được trên local / Colab / bất kỳ máy nào có internet.**
**Lưu ý:** Cần file `.env` chứa `WANDB_API_KEY` ở cùng thư mục repo.

In [ ]:
from pathlib import Path
NOTEBOOK = Path("__file__").resolve()
ROOT = NOTEBOOK
for _ in range(10):
    ROOT = ROOT.parent
    if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists(): break
import sys, os
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print(f"[ROOT] {ROOT}")

In [ ]:
from dotenv import load_dotenv
dotenv_path = ROOT / ".env"
load_dotenv(dotenv_path)
print(f"[.env] Loaded from {dotenv_path}")

import subprocess, sys
for pkg in ["wandb", "scikit-learn", "seaborn", "matplotlib", "timm", "albumentations"]:
    try: __import__(pkg.replace("-", "_"))
    except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

os.environ.setdefault("WANDB_API_KEY", os.getenv("WANDB_API_KEY", ""))
print(f"[wandb] API key: {'set' if os.getenv('WANDB_API_KEY') else 'MISSING'}")

In [ ]:
import wandb
api = wandb.Api()
print("[wandb] API ready")

## Fetch Runs from wandb

In [ ]:
PROJECT = "woodbench-taskB"
MODELS = ["convnext_tiny", "mobilenetv3_large", "swin_tiny", "resnet50", "efficientnet_b3"]
SEEDS = [42, 123, 456]

runs_data = {}

for model in MODELS:
    runs_data[model] = {}
    for seed in SEEDS:
        try:
            runs = api.runs(
                PROJECT,
                filters={
                    "$and": [
                        {"config.backbone": model},
                        {"config.seed": seed}
                    ]
                }
            )
            if runs:
                run = runs[0]
                summary = run.summary._json
                runs_data[model][seed] = {
                    "name": run.name,
                    "best_val_acc": summary.get("val/acc", 0),
                    "run_id": run.id,
                    "state": run.state,
                }
                print(f"[OK] {model} s{seed}: {runs_data[model][seed]['best_val_acc']:.2f}%")
            else:
                print(f"[MISSING] {model} s{seed}")
        except Exception as e:
            print(f"[ERR] {model} s{seed}: {e}")

## Summary Table

In [ ]:
import pandas as pd
import numpy as np

rows = []
for model in MODELS:
    for seed in SEEDS:
        d = runs_data[model].get(seed, {})
        rows.append({
            "Model": model,
            "Seed": seed,
            "Best Val Acc (%)": d.get("best_val_acc", 0),
            "State": d.get("state", "missing"),
        })

df = pd.DataFrame(rows)

summary = df.groupby("Model").agg(
    Mean_Acc=("Best Val Acc (%)", "mean"),
    Std_Acc=("Best Val Acc (%)", "std"),
    Min_Acc=("Best Val Acc (%)", "min"),
    Max_Acc=("Best Val Acc (%)", "max"),
).round(2)

print("\n" + "="*70)
print("TASK B — 5 MODELS × 3 SEEDS SUMMARY")
print("="*70)
print(df.to_string(index=False))
print("\n" + "-"*70)
print("PER-MODEL SUMMARY")
print("-"*70)
print(summary.to_string())

best_model = summary["Mean_Acc"].idxmax()
best_acc = summary.loc[best_model, "Mean_Acc"]
print(f"\nBest model: {best_model} ({best_acc:.2f}%)")

In [ ]:
out_dir = ROOT / "experiments" / "taskB"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "results_summary.csv", index=False)
summary.to_csv(out_dir / "results_per_model.csv")
print(f"[Saved] {out_dir / 'results_summary.csv'}")
print("\n" + df.to_markdown())
print("\n[Next] Run evaluate.py for full test-set metrics + confusion matrix")